# OctWave3 — Tom & Jerry Classification (single notebook)

**Runtime → Change runtime type → T4 GPU** before running anything.

Everything lives here: setup → train → **download submission** → analysis → next step.
Real logic stays in `src/*.py`; this notebook drives it.

**It never submits to Kaggle automatically.** Section 9 writes the CSV and downloads it
to your computer. You upload it to Kaggle yourself, when you decide to.

**Protected baseline: `exp02_5fold_b0` = 0.812339 (rank #9).** Never overwrite it —
every new run uses a new `exp_name`.

| Section | Does | GPU |
|---|---|---|
| 1–5 | setup, data | no |
| 6 | inspect data | no |
| 7–8 | configure, train 5 folds | **yes** |
| 9 | generalisation check | no |
| 10 | predict + TTA → **download CSV** | yes |
| 11–13 | error analysis, decision-rule search, next step | no |

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Install packages

If Colab shows **RESTART SESSION**, click it, then continue from section 3.

In [ ]:
!pip install -q timm albumentations kaggle

## 3. Mount Drive

Checkpoints and OOF predictions live here, so a disconnect costs one epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/octwave3/outputs')
for sub in ('checkpoints', 'submissions', 'oof', 'figures'):
    (DRIVE_OUT / sub).mkdir(parents=True, exist_ok=True)
print(DRIVE_OUT)

## 4. Clone / pull the repo

Needs a `GH_TOKEN` Colab secret (🔑 icon) for a private repo:
GitHub → Settings → Developer settings → Fine-grained tokens → repo `OctWave3`,
**Contents: Read and write**. Toggle *Notebook access* ON.

In [ ]:
import os, sys, subprocess

REPO_DIR = '/content/OctWave3'
os.chdir('/content')                 # never operate from inside a dir we may delete

try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = None

URL = (f'https://{TOKEN}@github.com/sasindu345/OctWave3.git' if TOKEN
       else 'https://github.com/sasindu345/OctWave3.git')

def run(*args, cwd=None):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    msg = (r.stdout + r.stderr).strip()
    if TOKEN:
        msg = msg.replace(TOKEN, '***')
    if msg:
        print(msg)
    return r.returncode

if os.path.isdir(REPO_DIR + '/.git'):
    run('git', 'pull', '-q', URL, 'main', cwd=REPO_DIR)
else:
    if run('git', 'clone', '-q', URL, REPO_DIR) != 0:
        raise SystemExit('clone failed - check the GH_TOKEN secret')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
import importlib; importlib.invalidate_caches()
run('git', 'log', '--oneline', '-3')

## 5. Download the competition data

Skip this cell if `/content/data` already exists from earlier in the session.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()                      # pick kaggle.json
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

COMP = 'oct-wave-3-0-kaggle-challenge-02'
if not os.path.isdir('/content/data/images'):
    !kaggle competitions download -c $COMP -p /content/data
    !unzip -q -o '/content/data/*.zip' -d /content/data
    !if [ -f /content/data/images.zip ]; then unzip -q -o /content/data/images.zip -d /content/data; fi
    !rm -f /content/data/oct-wave-3-0-kaggle-challenge-02.zip
!ls /content/data

## 6. Inspect the data

Facts, not assumptions. Everything printed here is measured.

In [ ]:
import pandas as pd, numpy as np, cv2
from pathlib import Path
D = Path('/content/data')

tr = pd.read_csv(D / 'train.csv'); te = pd.read_csv(D / 'test.csv')
print(f'train {len(tr)} rows | test {len(te)} rows | columns {list(tr.columns)}')

vc = tr['appearance'].value_counts().sort_index()
names = {0: 'neither', 1: 'Tom only', 2: 'Jerry only', 3: 'both'}
print('\nclass balance:')
for k, v in vc.items():
    print(f'  {k} {names[k]:11s} {v:5d} ({100*v/len(tr):5.1f}%) ' + '#' * int(45 * v / vc.max()))
print(f'  imbalance {vc.max()/vc.min():.1f}x | rarest {vc.min()} -> ~{vc.min()//5} per val fold')

imgs = sorted(Path('/content/data').rglob('*.jpg'))
shapes = {}
for f in np.random.default_rng(0).choice(imgs, min(200, len(imgs)), replace=False):
    im = cv2.imread(str(f))
    if im is not None:
        shapes[im.shape] = shapes.get(im.shape, 0) + 1
print(f'\n{len(imgs)} images | shapes (200 sampled): {dict(sorted(shapes.items(), key=lambda x: -x[1])[:4])}')

## 7. Configure — E4: EfficientNet-B2 @ 300

**Protected champion: `exp02_5fold_b0` = 0.812339 (#9). Never overwritten.**

The change under test is **representation**: `b0 @ 224` → `b2 @ 300`. Batch size drops
to 24 only because 300px needs the VRAM — that is a hardware constraint, not a
second variable under test.

Everything else is held at the champion's settings: AdamW, lr 3e-4, wd 1e-4,
label smoothing 0.05, class weights on, seed 42, same folds, same augmentation,
macro-F1 checkpointing, early stopping, AMP, 5 folds, HFlip TTA, plain argmax.

**No class multipliers.** They scored 0.7466 on Kaggle vs the champion's 0.8123 — rejected.

⚠️ **Read before running:** `epochs 12 → 18` also changes the OneCycleLR *shape*, not
just the length (see section 16). exp03 therefore varied two things at once. Setting
18 here keeps E4 comparable to exp03 rather than to exp02 — a deliberate choice so the
B0-vs-B2 comparison is at matched schedule length.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from pathlib import Path
for m in ('src.config','src.utils','src.dataset','src.model','src.train','src.analysis','src.predict'):
    importlib.reload(importlib.import_module(m))
from src.config import cfg

cfg.data_dir = Path('/content/data')
cfg.out_dir  = DRIVE_OUT

cfg.exp_name = 'exp04_b2_300'          # unique name - champion untouched
CHAMPION_EXP = 'exp02_5fold_b0'        # protected, Kaggle 0.812339
PREV_EXP     = 'exp03_5fold_b0_e18'    # for diversity analysis

# --- the change under test ---
cfg.model_name = 'tf_efficientnet_b2'
cfg.img_size   = 300
cfg.batch_size = 24                    # VRAM constraint only; 16 + grad_accum 2 if OOM
cfg.grad_accum = 1

# --- held fixed at champion settings ---
cfg.epochs          = 18
cfg.lr              = 3e-4
cfg.weight_decay    = 1e-4
cfg.num_classes     = 4
cfg.metric          = 'macro_f1'
cfg.class_weights   = True
cfg.label_smoothing = 0.05
cfg.drop_rate       = 0.3
cfg.seed            = 42
cfg.early_stop_patience = 6

cfg.to_dict()

## 8. Train all 5 folds

~5–8 min per fold on a T4. Resumes automatically after a disconnect: re-run
sections 3, 4, 7, 8 and finished folds are skipped in seconds.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.train import run_fold
import numpy as np

FOLDS = [0, 1, 2, 3, 4]
scores = {}
for f in FOLDS:
    print(f'\n{"="*55}\nFOLD {f}\n{"="*55}')
    scores[f] = run_fold(cfg, fold=f)

v = np.array(list(scores.values()))
print(f'\n{"="*55}')
print(f'CV macro F1: {v.mean():.4f} +/- {v.std(ddof=1):.4f}')
print('per fold   :', {k: round(x, 4) for k, x in scores.items()})
print('\nBaseline exp02 was 0.6419 +/- 0.0153. Remember the 0.018 noise floor.')

## 9. Generalisation check — did it learn, or memorise?

Three pieces of evidence:

1. **`overfit_check`** per fold → HEALTHY / MILD / OVERFITTING / UNDERTRAINED
2. **Fold spread** — mean ± std. std > 0.02 means fold noise exceeds most gains
3. **Box plot** — the spread, drawn

If folds still return UNDERTRAINED, more epochs may still help. If they flip to
OVERFITTING, 18 was too many and the answer is augmentation, not epochs.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.analysis import overfit_check, cv_summary, plot_fold_box, plot_learning_curves, use_house_style
use_house_style()

log = pd.read_json(cfg.out_dir / 'run_log.jsonl', lines=True)
mine = log[log.exp == cfg.exp_name]

verdicts = {}
for f in sorted(mine.fold.unique()):
    verdicts[int(f)] = overfit_check(mine, cfg.exp_name, fold=int(f))['verdict']
    print()
print('verdicts:', verdicts)

s = cv_summary(cfg.out_dir, cfg.exp_name, sorted(mine.fold.unique().tolist()), metric='macro_f1')
print(f"\nCV macro F1 {s['mean']:.4f} +/- {s['std']:.4f} | per fold {np.round(s['scores'],4)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
plot_learning_curves(mine, cfg.exp_name, ax=ax[0])
plot_fold_box([s], ax=ax[1])
plt.tight_layout()
fig.savefig(cfg.out_dir / f'figures/{cfg.exp_name}_training.png', bbox_inches='tight', dpi=130)
plt.show()

## 10. Predict → submission → **download to your computer**

5 fold checkpoints, probability averaging, horizontal-flip TTA, argmax.

**No automatic Kaggle submission.** The file downloads to your machine; you upload it
to Kaggle yourself after reading the analysis below.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.dataset import build_test_dataframe
from src.predict import predict, make_submission

test_df = build_test_dataframe(cfg)
ckpts = sorted((cfg.out_dir / 'checkpoints').glob(f'{cfg.exp_name}_f*_best.pt'))
print(f'{len(test_df)} test images | {len(ckpts)} checkpoints:', [c.name for c in ckpts])
assert len(ckpts) == 5, 'expected 5 fold checkpoints - train all folds first'

probs_test = predict(cfg, ckpts, test_df, tta=True)
sub = make_submission(cfg, probs_test, test_df, f'{cfg.exp_name}.csv')
np.save(cfg.out_dir / f'{cfg.exp_name}_test_probs.npy', probs_test)   # for later blending
sub.head()

In [ ]:
# --- download the CSV to your computer (no Kaggle auto-submit) ---
from google.colab import files
sub_path = cfg.out_dir / 'submissions' / f'{cfg.exp_name}.csv'
print('downloading', sub_path)
files.download(str(sub_path))

## 11. Error analysis — where is macro F1 actually being lost?

Macro F1 averages all 4 classes equally, so one weak class caps the score. This
finds it, with confidence intervals — class 3 has only ~44 images per fold, so its
F1 is noisy and a point estimate alone is not evidence.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import numpy as np, matplotlib.pyplot as plt
from src.analysis import per_class_report, plot_confusion, plot_reliability, bootstrap_ci, use_house_style
from src.utils import load_oof
use_house_style()

FOLDS = [0, 1, 2, 3, 4]
classes = ['0 neither', '1 Tom', '2 Jerry', '3 both']

oof_p = np.concatenate([load_oof(cfg.out_dir, cfg.exp_name, f)[0] for f in FOLDS])
oof_t = np.concatenate([load_oof(cfg.out_dir, cfg.exp_name, f)[1] for f in FOLDS])
print(f'pooled OOF: {len(oof_t)} samples\n')

per_class_report(oof_t, oof_p, classes)

ci = bootstrap_ci(oof_t, oof_p, metric='macro_f1')
print(f"\npooled OOF macro F1 {ci['point']:.4f}  95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
plot_confusion(oof_t, oof_p, classes, ax=ax[0])
plot_reliability(oof_t, oof_p, ax=ax[1])
plt.tight_layout()
fig.savefig(cfg.out_dir / f'figures/{cfg.exp_name}_errors.png', bbox_inches='tight', dpi=130)
plt.show()

print('\nREAD THE CONFUSION MATRIX: the hottest off-diagonal cell is your biggest')
print('single failure mode, and it points at a targeted fix rather than "train longer".')

## 12. Is B2 complementary to B0?

Raw score is not what decides an ensemble — **different errors** are. A weaker model
still helps if it is right where the stronger one is wrong.

The number to read is the **oracle ceiling**: accuracy if you always picked whichever
model was correct. If that ceiling sits barely above the better single model, there is
nothing for a blend to recover and you should stop here.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import numpy as np
from src.analysis import ensemble_diversity
from src.utils import load_oof

FOLDS = [0, 1, 2, 3, 4]

def pooled(exp):
    p = np.concatenate([load_oof(cfg.out_dir, exp, f)[0] for f in FOLDS])
    t = np.concatenate([load_oof(cfg.out_dir, exp, f)[1] for f in FOLDS])
    return p, t

try:
    champ_p, champ_t = pooled(CHAMPION_EXP)
    b2_p,   b2_t     = pooled(cfg.exp_name)
    assert (champ_t == b2_t).all(), 'fold splits differ - comparison invalid'
    div = ensemble_diversity(champ_t, champ_p, b2_p, CHAMPION_EXP, cfg.exp_name)
except FileNotFoundError as e:
    print('missing OOF, cannot compare:', e)

## 13. Blend weight search — scored per fold, not pooled

`w * B0 + (1-w) * B2`, evaluated on each fold separately.

Per-fold on purpose. The class-multiplier failure (OOF said better, Kaggle said
−0.066) is a warning that anything tuned on pooled OOF can fail to transfer. A weight
worth submitting should win on **4 or 5 individual folds**, and by more than the
**0.018 noise floor** — not just nudge the pooled mean.

Coarse grid only. Fine-grained weight search is just another way to overfit OOF.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.analysis import blend_search

try:
    bdf = blend_search(cfg.out_dir, CHAMPION_EXP, cfg.exp_name, FOLDS,
                       weights=[1.0, 0.7, 0.6, 0.5, 0.4, 0.3, 0.0], metric='macro_f1')
except FileNotFoundError as e:
    print('missing OOF:', e)

## 14. Build the submission you actually intend to send

Three candidates. Pick **one** based on the evidence above, set `CHOICE`, run, download.

- `'b2'` — B2 alone. Only if it clearly beats the champion.
- `'blend'` — B0+B2 at the chosen weight. Only if section 13 showed 4+ folds winning.
- `'none'` — evidence is weak. **Keep the 0.812339 champion.** This is a valid, common outcome.

Requires the champion's test probabilities at
`outputs/exp02_5fold_b0_test_probs.npy`. If that file is missing, re-run inference for
the champion checkpoints first — do **not** retrain them.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import numpy as np
from pathlib import Path
from src.dataset import build_test_dataframe
from src.predict import predict, make_submission

CHOICE  = 'none'      # 'b2' | 'blend' | 'none'
W_B0    = 0.6         # only used when CHOICE == 'blend'

test_df = build_test_dataframe(cfg)

if CHOICE == 'none':
    print('No submission built. Champion 0.812339 stays as the final answer.')
else:
    ck = sorted((cfg.out_dir / 'checkpoints').glob(f'{cfg.exp_name}_f*_best.pt'))
    assert len(ck) == 5, f'expected 5 checkpoints, found {len(ck)}'
    b2_test = predict(cfg, ck, test_df, tta=True)
    np.save(cfg.out_dir / f'{cfg.exp_name}_test_probs.npy', b2_test)

    if CHOICE == 'b2':
        final, tag = b2_test, cfg.exp_name
    else:
        champ_file = cfg.out_dir / f'{CHAMPION_EXP}_test_probs.npy'
        if not champ_file.exists():
            raise FileNotFoundError(
                f'{champ_file} missing - re-run inference with the champion checkpoints '
                '(do NOT retrain them) to regenerate it.')
        champ_test = np.load(champ_file)
        final = W_B0 * champ_test + (1 - W_B0) * b2_test
        tag = f'blend_b0{int(W_B0*100)}_b2{int((1-W_B0)*100)}'

    sub = make_submission(cfg, final, test_df, f'{tag}.csv')
    from google.colab import files
    files.download(str(cfg.out_dir / 'submissions' / f'{tag}.csv'))
    print(f'\nDownloaded {tag}.csv - upload to Kaggle manually.')

## 13. Is this run actually better than the protected baseline?

Paired bootstrap + McNemar on the **same** validation samples. Two headline scores
are not a comparison.

`ADOPT` means the new run beat `exp02_5fold_b0` by more than noise. Anything else and
**the 0.812339 submission stays as your final answer.**

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

import numpy as np
from src.analysis import decide, cv_summary, plot_fold_box, use_house_style
from src.utils import load_oof
import matplotlib.pyplot as plt
use_house_style()

try:
    a_p = np.concatenate([load_oof(cfg.out_dir, BASELINE_EXP, f)[0] for f in FOLDS])
    a_t = np.concatenate([load_oof(cfg.out_dir, BASELINE_EXP, f)[1] for f in FOLDS])
    b_p = np.concatenate([load_oof(cfg.out_dir, cfg.exp_name, f)[0] for f in FOLDS])

    assert (a_t == oof_t).all(), 'fold split changed - comparison invalid'
    result = decide(a_t, a_p, b_p, BASELINE_EXP, cfg.exp_name, metric='macro_f1')

    sums = [cv_summary(cfg.out_dir, e, FOLDS, metric='macro_f1') for e in (BASELINE_EXP, cfg.exp_name)]
    for s in sums:
        print(f"  {s['exp']}: {s['mean']:.4f} +/- {s['std']:.4f}")
    plot_fold_box(sums); plt.show()
except FileNotFoundError as e:
    print('baseline OOF not in Drive, cannot compare:', e)

## 14. Push results back to GitHub

Sends the run log and figures back to the repo (small files only — checkpoints and
raw predictions stay in Drive). Then `git pull` on your laptop to read them.

In [ ]:
import shutil, subprocess, os
from pathlib import Path
from google.colab import userdata

REPO = Path('/content/OctWave3')
RESULTS = REPO / 'results'
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)

log_src = DRIVE_OUT / 'run_log.jsonl'
if log_src.exists():
    shutil.copy(log_src, RESULTS / 'run_log.jsonl')
for png in (DRIVE_OUT / 'figures').glob('*.png'):
    shutil.copy(png, RESULTS / 'figures' / png.name)
print('staged:', [p.name for p in RESULTS.rglob('*') if p.is_file()])

TOKEN = userdata.get('GH_TOKEN')

def git(*args, secret=False):
    r = subprocess.run(('git',) + args, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret and TOKEN:
        out = out.replace(TOKEN, '***')
    if out:
        print(out)
    return r.returncode

git('config', 'user.email', 'cryptxgmora@gmail.com')
git('config', 'user.name', 'sasindu345')
git('add', 'results', 'logs')
if git('diff', '--cached', '--quiet') == 0:
    print('nothing new to push')
else:
    git('commit', '-m', f'results: {cfg.exp_name}')
    git('push', f'https://{TOKEN}@github.com/sasindu345/OctWave3.git', 'HEAD:main', secret=True)
    print('pushed')

## 15. Decide the next step

Read your own output above, in this order:

**a) Section 13 verdict**
- `ADOPT` → this run beat the baseline. Submit its CSV, record the LB score, make it the new baseline.
- `WEAK` / `REJECT` → **keep 0.812339.** Longer training was not the answer.

**b) Section 9 verdicts**
- still `UNDERTRAINED` → the schedule is still too short; try 24 epochs
- now `OVERFITTING` → 18 was too many; go back to 12 and add augmentation strength instead

**c) Section 11 confusion matrix** — the hottest off-diagonal cell names the next fix:
- 3 confused with 1 or 2 → "both" is hard to separate; **resolution** is the likely lever (Jerry is small in an 854px frame downscaled to 224)
- 0 confused with everything → the model is guessing on empty frames; more augmentation
- one class with high recall but low precision → the class weights are over-correcting; test `class_weights=False`

**d) Section 12 verdict** — if `ADOPT`, you gained score with no training at all.

**Then run ONE experiment, not five.** Change a single variable, keep a new `exp_name`,
and let section 13 decide. Log it in `logs/EXPERIMENTS.md` and `logs/DECISIONS.md`.

**Never delete or overwrite the 0.812339 submission.**